In [ ]:
import pandas as pd
import numpy as np
import os

# --- 1. FOLDER SETUP ---
for folder in ['data/raw', 'data/chunks', 'data/processed', 'models']:
    os.makedirs(folder, exist_ok=True)

# --- 2. SEASONAL WEIGHTED SAMPLING ---
INPUT_FILE = 'data/raw/filtered_2024_2026.csv' 
OUTPUT_SAMPLE = 'data/raw/weighted_sample_1lakh.csv'

print("📂 Loading original archive...")
df = pd.read_csv(INPUT_FILE)
df.columns = [c.lower() for c in df.columns]
df['acq_date'] = pd.to_datetime(df['acq_date'])
df['month'] = df['acq_date'].dt.month

# Weighting: Give 5x more priority to Summer (Feb-June)
df['weight'] = df['month'].apply(lambda x: 5.0 if 2 <= x <= 6 else 1.0)

print("⚖️ Generating diversified 1-lakh row sample...")
df_sampled = df.sample(n=100000, weights='weight', random_state=42).copy()
df_sampled.to_csv(OUTPUT_SAMPLE, index=False)
print(f"✅ Sample saved to {OUTPUT_SAMPLE}")

In [ ]:
import pandas as pd
import requests
import time
import os

# --- CONFIG ---
SAMPLE_FILE = 'data/raw/weighted_sample_1lakh.csv'
FINISH_FILE = 'data/raw/enriched_weather_data.csv'
API_URL = "https://archive-api.open-meteo.com/v1/archive"

df = pd.read_csv(SAMPLE_FILE)
df['acq_date'] = pd.to_datetime(df['acq_date']).dt.strftime('%Y-%m-%d')

# Handle time: Converts '930' or '1445' to integer hour (0-23)
df['hour_idx'] = df['acq_time'].apply(lambda x: int(str(int(x)).zfill(4)[:2]) if pd.notnull(x) else 12)

# Checkpoint Logic
if os.path.exists(FINISH_FILE):
    done_df = pd.read_csv(FINISH_FILE)
    done_keys = set(done_df['latitude'].astype(str) + done_df['acq_date'].astype(str))
else:
    done_keys = set()
    df.head(0).to_csv(FINISH_FILE, index=False)

# Start Batching (50 locations per call)
grouped = df.groupby('acq_date')
print("🚀 Starting Weather Retrieval (Matched to acq_time)...")

for date_str, group in grouped:
    to_proc = group[~(group['latitude'].astype(str) + date_str).isin(done_keys)]
    if to_proc.empty: continue

    for j in range(0, len(to_proc), 50):
        batch = to_proc.iloc[j : j + 50]
        params = {
            "latitude": ",".join(map(str, batch['latitude'])),
            "longitude": ",".join(map(str, batch['longitude'])),
            "start_date": date_str, "end_date": date_str,
            "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m"
        }
        
        try:
            r = requests.get(API_URL, params=params, timeout=60)
            if r.status_code == 200:
                data = r.json()
                res_list = data if isinstance(data, list) else [data]
                enriched = []
                for idx, (_, row) in enumerate(batch.iterrows()):
                    w = res_list[idx]['hourly']
                    h = row['hour_idx']
                    enriched.append({**row.to_dict(), 
                                     'temp': w['temperature_2m'][h], 
                                     'humidity': w['relative_humidity_2m'][h], 
                                     'wind': w['wind_speed_10m'][h]})
                pd.DataFrame(enriched).to_csv(FINISH_FILE, mode='a', header=False, index=False)
            elif r.status_code == 429:
                time.sleep(60)
        except: pass
        time.sleep(1.2)
    print(f"✅ Day {date_str} Done", end='\r')